# 297. Serialize and Deserialize Binary Tree

Design an algorithm to serialize a binary tree into a string and deserialize the string back into a tree.

## Problem Statement

You need to:
1. **Serialize**: Convert a binary tree into a string representation
2. **Deserialize**: Reconstruct the binary tree from the string representation

The string format and tree node definition are flexible. You just need to ensure that a binary tree can be serialized to a string and this string can be deserialized to the original tree structure.

## Example

```
    1
   / \
  2   3
     / \
    4   5
```

**Serialized:** "1,2,null,null,3,4,null,null,5,null,null"
**Deserialized:** Back to the original tree

## Key Insight: The Iterator Pattern

The "Aha! Moment" for Pythonistas:

| Concept | NeetCode Logic | Pythonic Logic (iter) |
|---------|----------------|----------------------|
| Get next value | `val = data.pop(0)` → **O(N)** ❌ | `val = next(it)` → **O(1)** ✅ |
| Manual tracking | `self.i += 1` | Iterator advances automatically |
| Recursion overhead | `dfs(data[1:])` (copies list) | `dfs(it)` (reuses stream) |

**Translation for your mind:**
When the video says _"we remove the first element and go to the next"_, immediately think: **`val = next(it)`**


In [ ]:
from typing import Optional, List
from collections import deque
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


## Understanding Binary Tree Serialization Concepts

### How DFS "Consumes" the Data Stream

**Pre-order traversal** (root → left → right) is perfect for both serialization and deserialization:

```
        1
       / \
      2   3
     /   / \
    4   5   6

Serialization (DFS pre-order): 1, 2, 4, null, null, null, 3, 5, null, null, 6, null, null
                              (root, left subtree, right subtree for each node)
```

### The Problem with `data.pop(0)`: O(N) per operation ❌

```python
# Naive approach - SLOW!
def deserialize(data):
    data_list = data.split(',')
    
    def dfs():
        val = data_list.pop(0)  # ⚠️ O(N) - shifts entire list!
        if val == 'null':
            return None
        node = TreeNode(int(val))
        node.left = dfs(data_list[1:])   # ⚠️ Copies list - O(N) space!
        node.right = dfs(data_list)
        return node
    
    return dfs()
```

**Why is `pop(0)` slow?**
- Removes first element → **shifts all remaining elements** one position left
- For a list of N elements, this is **O(N) operations**
- Total complexity: **O(N²)** 😱

### The Pythonic Solution: `next(it)` — O(1) ✅

```python
# Senior approach - FAST!
def deserialize(data):
    data_list = data.split(',')
    it = iter(data_list)  # Create iterator
    
    def dfs():
        val = next(it)  # ✅ O(1) - just advances pointer!
        if val == 'null':
            return None
        node = TreeNode(int(val))
        node.left = dfs()   # ✅ Same iterator, auto-advances!
        node.right = dfs()
        return node
    
    return dfs()
```

**Why is `next(it)` fast?**
- Iterator maintains an **internal pointer**
- `next()` just **returns current value and advances pointer** → **O(1)**
- No copying, no shifting, no manual tracking
- Total complexity: **O(N)** ✅


## Naive Approach: Using `list.pop(0)` — O(N²) ❌

This is the universal approach taught in most courses (like NeetCode).

**Time Complexity:** O(N²) - each `pop(0)` is O(N)
**Space Complexity:** O(N) - for the list


In [ ]:
class CodecNaive:
    """NeetCode Universal Approach - Using pop(0)"""
    
    def serialize(self, root: Optional[TreeNode]) -> str:
        """Convert tree to string using pre-order DFS"""
        result = []
        
        def dfs(node):
            if not node:
                result.append("null")
                return
            
            result.append(str(node.val))
            dfs(node.left)
            dfs(node.right)
        
        dfs(root)
        return ",".join(result)
    
    def deserialize(self, data: str) -> Optional[TreeNode]:
        """Reconstruct tree from string - SLOW with pop(0)"""
        data_list = data.split(",")
        
        def dfs():
            val = data_list.pop(0)  # ⚠️ O(N) operation!
            if val == "null":
                return None
            
            node = TreeNode(int(val))
            node.left = dfs()
            node.right = dfs()
            return node
        
        return dfs()


# Test the naive approach
print("=" * 60)
print("NAIVE APPROACH: Using pop(0)")
print("=" * 60)

codec_naive = CodecNaive()

# Build test tree
root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.right.left = TreeNode(4)
root.right.right = TreeNode(5)

serialized = codec_naive.serialize(root)
print(f"Serialized: {serialized}")

deserialized = codec_naive.deserialize(serialized)
print(f"Reserialized: {codec_naive.serialize(deserialized)}")
print(f"✅ Naive approach works, but is O(N²) for deserialization!\n")


## Optimized Approach: Using Iterator with `next()` — O(N) ✅

**The Pythonic Trick:** Replace manual index tracking with Python's iterator pattern.

**Time Complexity:** O(N) - each value processed exactly once
**Space Complexity:** O(N) - for the iterator

### Key Differences:

| Aspect | `pop(0)` | `next(it)` |
|--------|---------|-----------|
| **Operation** | Removes from list | Advances pointer |
| **Complexity** | O(N) per call | O(1) per call |
| **Manual index?** | No, but costly | No, automatic |
| **Total deserialize** | O(N²) | O(N) |

**Why this works:**
1. `it = iter(data_list)` creates an iterator object
2. `next(it)` returns current element and **automatically advances**
3. The iterator is **shared across all recursive calls** via closure
4. No list copying, no shifting - just pointer movement


In [ ]:
class CodecIterator:
    """Pythonic Approach - Using iter() and next()"""
    
    def serialize(self, root: Optional[TreeNode]) -> str:
        """Convert tree to string using pre-order DFS"""
        result = []
        
        def dfs(node):
            if not node:
                result.append("null")
                return
            
            result.append(str(node.val))
            dfs(node.left)
            dfs(node.right)
        
        dfs(root)
        return ",".join(result)
    
    def deserialize(self, data: str) -> Optional[TreeNode]:
        """Reconstruct tree from string - FAST with iterator"""
        data_list = data.split(",")
        it = iter(data_list)  # ✅ Create iterator once
        
        def dfs():
            val = next(it)  # ✅ O(1) - get current, auto-advance!
            if val == "null":
                return None
            
            node = TreeNode(int(val))
            node.left = dfs()   # ✅ Same iterator, keeps advancing
            node.right = dfs()
            return node
        
        return dfs()


# Test the iterator approach
print("=" * 60)
print("ITERATOR APPROACH: Using next(it)")
print("=" * 60)

codec_iter = CodecIterator()

# Build test tree
root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.right.left = TreeNode(4)
root.right.right = TreeNode(5)

serialized = codec_iter.serialize(root)
print(f"Serialized: {serialized}")

deserialized = codec_iter.deserialize(serialized)
print(f"Reserialized: {codec_iter.serialize(deserialized)}")
print(f"✅ Iterator approach is O(N) and much cleaner!\n")


## Alternative: Using `deque.popleft()` — O(N) ✅

**Time Complexity:** O(N) - `popleft()` is O(1)
**Space Complexity:** O(N) - for the deque

This is faster than `list.pop(0)` but still not as elegant as the iterator approach.

| Method | Time | Notes |
|--------|------|-------|
| `list.pop(0)` | O(N²) | Shifts entire list ❌ |
| `deque.popleft()` | O(N) | O(1) per operation ✅ |
| `iter() + next()` | O(N) | Most Pythonic ✅ |


In [ ]:
class CodecDeque:
    """Using deque.popleft() - O(1) per operation"""
    
    def serialize(self, root: Optional[TreeNode]) -> str:
        """Convert tree to string using pre-order DFS"""
        result = []
        
        def dfs(node):
            if not node:
                result.append("null")
                return
            
            result.append(str(node.val))
            dfs(node.left)
            dfs(node.right)
        
        dfs(root)
        return ",".join(result)
    
    def deserialize(self, data: str) -> Optional[TreeNode]:
        """Reconstruct tree from string using deque"""
        data_deque = deque(data.split(","))
        
        def dfs():
            val = data_deque.popleft()  # ✅ O(1) - optimized removal
            if val == "null":
                return None
            
            node = TreeNode(int(val))
            node.left = dfs()
            node.right = dfs()
            return node
        
        return dfs()


# Test the deque approach
print("=" * 60)
print("DEQUE APPROACH: Using popleft()")
print("=" * 60)

codec_deque = CodecDeque()

# Build test tree
root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.right.left = TreeNode(4)
root.right.right = TreeNode(5)

serialized = codec_deque.serialize(root)
print(f"Serialized: {serialized}")

deserialized = codec_deque.deserialize(serialized)
print(f"Reserialized: {codec_deque.serialize(deserialized)}")
print(f"✅ Deque approach is O(N) with O(1) popleft()!\n")


## Alternative: Using Global Index — O(N) ✅

**Manual index tracking approach.** Not as Pythonic as iterator, but still O(N).

| Feature | Global Index | Iterator |
|---------|--------------|----------|
| **Manual tracking?** | Yes (self.i) | No (automatic) |
| **Code clarity** | ⚠️ More verbose | ✅ Cleaner |
| **Performance** | O(N) | O(N) |
| **Pythonic** | ❌ | ✅ |


In [ ]:
class CodecGlobalIndex:
    """Using global index - Manual tracking"""
    
    def serialize(self, root: Optional[TreeNode]) -> str:
        """Convert tree to string using pre-order DFS"""
        result = []
        
        def dfs(node):
            if not node:
                result.append("null")
                return
            
            result.append(str(node.val))
            dfs(node.left)
            dfs(node.right)
        
        dfs(root)
        return ",".join(result)
    
    def deserialize(self, data: str) -> Optional[TreeNode]:
        """Reconstruct tree from string using manual index"""
        data_list = data.split(",")
        self.i = 0  # ⚠️ Global index (mutable state)
        
        def dfs():
            val = data_list[self.i]  # Get current value
            self.i += 1  # ⚠️ Manual increment
            
            if val == "null":
                return None
            
            node = TreeNode(int(val))
            node.left = dfs()
            node.right = dfs()
            return node
        
        return dfs()


# Test the global index approach
print("=" * 60)
print("GLOBAL INDEX APPROACH: Using manual self.i")
print("=" * 60)

codec_idx = CodecGlobalIndex()

# Build test tree
root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.right.left = TreeNode(4)
root.right.right = TreeNode(5)

serialized = codec_idx.serialize(root)
print(f"Serialized: {serialized}")

deserialized = codec_idx.deserialize(serialized)
print(f"Reserialized: {codec_idx.serialize(deserialized)}")
print(f"✅ Global index approach works, but requires manual state tracking\n")


## Performance Comparison: All Approaches

**Visual Flow Comparison:**

### NeetCode (pop(0)) - SLOW ❌
```
data_list = [1, 2, null, null, 3, 4, null, ...]
              ↑
        ❌ pop(0) shifts entire list left!
        ❌ O(N) operation
        ❌ Repeated N times = O(N²) total
```

### Iterator (next) - FAST ✅
```
data_list = [1, 2, null, null, 3, 4, null, ...]
            ↑
        ✅ next() just advances pointer
        ✅ O(1) operation
        ✅ Repeated N times = O(N) total
        
it = iter(data_list)
val = next(it)  ← pointer moves automatically
```

### Summary Table

| Approach | Time | Space | Clarity | Notes |
|----------|------|-------|---------|-------|
| **pop(0)** | O(N²) | O(N) | ⚠️ Simple | ❌ Slow - shifts list |
| **deque.popleft()** | O(N) | O(N) | ⚠️ Good | ✅ Fast enough |
| **Global index** | O(N) | O(N) | ❌ Poor | ⚠️ Manual state |
| **Iterator + next()** | O(N) | O(N) | ✅ Best | ✅ Pythonic, fast |


In [ ]:
# Performance benchmark
print("=" * 60)
print("PERFORMANCE BENCHMARK")
print("=" * 60)

# Build a larger tree for benchmarking
def build_deep_tree(depth):
    """Build a complete binary tree of given depth"""
    def build(d):
        if d == 0:
            return None
        node = TreeNode(d)
        node.left = build(d - 1)
        node.right = build(d - 1)
        return node
    return build(depth)

# Test with depth 10 (1023 nodes)
large_tree = build_deep_tree(10)

# Serialize once (all approaches have the same serialize)
large_serialized = CodecIterator().serialize(large_tree)
print(f"Tree size: {len(large_serialized.split(','))} elements\n")

# Benchmark each approach
approaches = [
    ("pop(0) - Naive", CodecNaive()),
    ("deque.popleft()", CodecDeque()),
    ("Global Index", CodecGlobalIndex()),
    ("Iterator + next() ⭐", CodecIterator()),
]

results = []
for name, codec in approaches:
    start = time.time()
    for _ in range(10):  # Run 10 times
        codec.deserialize(large_serialized)
    elapsed = time.time() - start
    results.append((name, elapsed))
    print(f"{name:25} → {elapsed:.4f}s (10 iterations)")

print("\n" + "=" * 60)
print(f"🏆 Winner: {min(results, key=lambda x: x[1])[0]}")
print("=" * 60)


## Test Cases and Validation

Validate all approaches with various tree configurations.


In [ ]:
def trees_equal(n1: Optional[TreeNode], n2: Optional[TreeNode]) -> bool:
    """Check if two trees are structurally identical"""
    if not n1 and not n2:
        return True
    if not n1 or not n2:
        return False
    return (n1.val == n2.val and 
            trees_equal(n1.left, n2.left) and 
            trees_equal(n1.right, n2.right))

def test_codec(codec, name):
    """Test a codec with various cases"""
    print(f"\n{'=' * 60}")
    print(f"Testing: {name}")
    print('=' * 60)
    
    test_cases = [
        ("Single node", TreeNode(1)),
        ("Empty tree", None),
        ("Left-skewed tree", TreeNode(1, TreeNode(2, TreeNode(3)))),
        ("Right-skewed tree", TreeNode(1, right=TreeNode(2, right=TreeNode(3)))),
    ]
    
    # Add balanced tree
    balanced = TreeNode(1)
    balanced.left = TreeNode(2)
    balanced.right = TreeNode(3)
    balanced.left.left = TreeNode(4)
    balanced.left.right = TreeNode(5)
    balanced.right.left = TreeNode(6)
    balanced.right.right = TreeNode(7)
    test_cases.append(("Balanced tree (depth 2)", balanced))
    
    for description, root in test_cases:
        serialized = codec.serialize(root)
        deserialized = codec.deserialize(serialized)
        match = trees_equal(root, deserialized)
        status = "✅ PASS" if match else "❌ FAIL"
        print(f"{status} | {description}")
        if not match:
            print(f"       Original: {codec.serialize(root)}")
            print(f"       Result:   {codec.serialize(deserialized)}")

# Test all approaches
for name, codec in approaches:
    test_codec(codec, name)

print(f"\n{'=' * 60}")
print("🎓 KEY TAKEAWAY FOR PYTHONISTAS")
print('=' * 60)
print("""
When you see "remove the first element and go to the next":
  ❌ SLOW: val = data.pop(0)           → O(N²) total
  ✅ FAST: val = next(iter(data_list)) → O(N) total
  
The iterator pattern is YOUR secret weapon in Python!
It's not just fast, it's elegant and Pythonic. Use it everywhere.
""")
